# 04 — DistilBERT


In [ ]:
# Detect the environment and configure the project
from pathlib import Path
import os
import sys

# Detect if we're running in Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab")

    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/voxforge-ai-review-analytics"
    )

else:
    print("Running in VS Code / Local")

    PROJECT_ROOT = Path.cwd()

    while PROJECT_ROOT != PROJECT_ROOT.parent:
        if (PROJECT_ROOT / "src").exists():
            break
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Install dependencies (Colab only)
if IN_COLAB:
    !pip install -q -r requirements.txt

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import *

from src.config import (
    CLEANED_REVIEWS_PATH, CONFUSION_MATRICES_DIR, MODEL_TRACKING_PATH,
    PLOTS_DIR, PREDICTIONS_DIR, RANDOM_STATE, SENTIMENT_MODELS_DIR, create_project_directories
)
from src.data.load import load_csv
from src.sentiment.transformer import (
    TransformerExperimentConfig, evaluate_trained_transformer,
    plot_training_history, prepare_transformer_sentiment_data,
    runtime_device, save_transformer_artifact, save_transformer_predictions,
    train_transformer,
)
from src.tracker import log_experiment

create_project_directories()
print("Runtime device:", runtime_device())

Runtime device: cuda


In [6]:
# model constants
MODEL_DIR = SENTIMENT_MODELS_DIR / "distilbert_sentiment"
CONFUSION_PATH = CONFUSION_MATRICES_DIR / "distilbert_sentiment.png"
HISTORY_PATH = PLOTS_DIR / "distilbert_sentiment_training_history.png"
PREDICTIONS_PATH = PREDICTIONS_DIR / "distilbert_sentiment_validation.csv"

In [7]:
# Load the cleaned dataset
reviews = load_csv(CLEANED_REVIEWS_PATH)
model_df = prepare_transformer_sentiment_data(reviews)

FileNotFoundError: [Errno 2] No such file or directory: '/content/voxforge-ai-review-analytics/data/processed/cleaned_reviews.csv'

In [ ]:
#  split the cleaned dataset
X_train_val, X_test, y_train_val, y_test = train_test_split(
    model_df["transformer_text"], model_df["sentiment_label"],
    test_size=0.20, random_state=RANDOM_STATE, stratify=model_df["sentiment_label"],
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=RANDOM_STATE, stratify=y_train_val,
)
print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Test held out: {len(X_test):,}")

Train: 30,258 | Validation: 7,565 | Test held out: 9,456


In [ ]:
# configure and train
config = TransformerExperimentConfig(
    model_id="sentiment_02",
    model_name="DistilBERT",
    pretrained_model="distilbert-base-uncased",
    output_dir=MODEL_DIR,
    max_length=256,
    epochs=3,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5,
    gradient_accumulation_steps=1,
    use_lora=False,
    use_4bit=False,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    lora_target_modules=("q_lin", "v_lin"),
)
config

TransformerExperimentConfig(model_id='sentiment_02', model_name='DistilBERT', pretrained_model='distilbert-base-uncased', output_dir=PosixPath('/Users/karima/Ironhack-challenges/voxforge-ai-review-analytics/models/sentiment/distilbert_sentiment'), max_length=256, epochs=3, train_batch_size=16, eval_batch_size=32, learning_rate=2e-05, weight_decay=0.01, warmup_ratio=0.1, gradient_accumulation_steps=1, use_lora=False, use_4bit=False, lora_r=16, lora_alpha=32, lora_dropout=0.1, lora_target_modules=('q_lin', 'v_lin'), random_state=42, early_stopping_patience=2)

In [ ]:
run = train_transformer(
    config, X_train, y_train, X_val, y_val
)

/Users/karima/Ironhack-challenges/voxforge-ai-review-analytics/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/5676 [00:00<?, ?it/s]

RuntimeError: MPS backend out of memory (MPS allocated: 2.87 GB, other allocations: 3.91 GB, max allowed: 6.80 GB). Tried to allocate 48.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
# Evaluate and save useful outputs
validation = evaluate_trained_transformer(
    run, X_val, y_val, confusion_matrix_path=CONFUSION_MATRICES_DIR
)
display(pd.DataFrame([validation["metrics"]]))
display(validation["classification_report"])

In [ ]:
saved_model = save_transformer_artifact(run, MODEL_DIR)
saved_predictions = save_transformer_predictions(
    X_val, y_val, validation, PREDICTIONS_DIR
)
saved_history = plot_training_history(run, HISTORY_PATH)
print("Model:", saved_model)
print("Predictions:", saved_predictions)
print("Training plot:", saved_history)
print("Confusion matrix:", CONFUSION_MATRICES_DIR)

In [ ]:
# Update the shared model tracker
tracking = log_experiment(
    model_id=config.model_id,
    model_name=config.model_name,
    model_family="Transformer",
    features="Contextual token embeddings",
    preprocessing="transformer_text; tokenizer truncation and dynamic padding",
    algorithm="Full fine-tuning",
    pretrained_model=config.pretrained_model,
    dataset=CLEANED_REVIEWS_PATH.name,
    training_rows=len(X_train),
    validation_rows=len(X_val),
    metrics=validation["metrics"],
    training_time_seconds=run["training_time_seconds"],
    inference_time_ms=validation["inference"]["average_inference_ms"],
    epochs=config.epochs,
    batch_size=config.train_batch_size,
    learning_rate=config.learning_rate,
    max_length=config.max_length,
    artifact_path=saved_model,
    hyperparameters={**config.to_dict(), "using_4bit": run["using_4bit"]},
    output_file=MODEL_TRACKING_PATH,
)
display(tracking)

The test split remains untouched. Use it only once after selecting the final model from validation metrics.